# Lounge scorecard: served demand + cost

**What changed from v3:** demand is served women (not catchment women); capture and rating left the scorecard (rating now sits inside the Huff attractiveness); cost efficiency joined; cannibalisation is a display-only what-if. Composite ≥ 0.65 is PROTECT, ≤ 0.35 SHRINK. Reference: the frozen v3 labels in `data/scenarios/baseline_v3_labels.csv`.

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = site cost** (`src/features/lounges.py`): shop rent × 150 m² + an assumed fit-out written off over 5 years. Cost efficiency = served women per AED 1k of site cost; `cost_index` = that ÷ the emirate's median lounge (1 = typical).
3. **Lounge scorecard** = served demand + cost index. Cannibalisation (the share of a lounge's women its siblings would absorb if it closed) is shown, not scored.
4. **Growth** = what-if: open a typical lounge (median lounge reviews and rating) at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Signals and anchors

In [2]:
pd.DataFrame([dict(signal=s.name, field=s.field, worst=s.worst, best=s.best, weight=s.weight, why=s.why) for s in scorecard.SIGNALS])

,signal,field,worst,best,weight,why
0,demand,served_women,0.0,20000.0,1.0,Quick first cut (2026-10-11): served women run...
1,cost,cost_index,0.5,1.5,1.0,1 = as cost-efficient as the emirate's median ...


## 2. v3 → new calls, with reasons

In [3]:
rows = []
for b, d in D.items():
    if d.action == "NOT SCORED": continue
    x = F.loc[b]
    reason = (f"serves {x.served_women:,.0f} (demand {d.scores['demand']:.2f}); "
              + (f"cost index {x.cost_index:.2f} (cost {d.scores['cost']:.2f})" if x.cost_index == x.cost_index and x.cost_index is not None
                 else "no rent: cost neutral 0.5"))
    rows.append(dict(lounge=b, emirate=x.emirate, v3=old_action.get(b), new=d.action, composite=round(d.composite, 3),
                     confidence=d.confidence, reason=reason, weight_flips="; ".join(d.weight_flips)))
t = pd.DataFrame(rows).sort_values("composite", ascending=False).set_index("lounge")
print(t.new.value_counts().to_dict()); print(pd.crosstab(t.v3, t.new))
with pd.option_context("display.max_colwidth", 90): display(t)

{'HOLD': 13, 'SHRINK': 7, 'PROTECT': 3}
new      HOLD  PROTECT  SHRINK
v3                            
HOLD       10        2       3
PROTECT     2        1       0
SHRINK      1        0       4


,emirate,v3,new,composite,confidence,reason,weight_flips
lounge,,,,,,,
al-dhafra,Abu Dhabi,HOLD,PROTECT,0.941,high,"serves 17,634 (demand 0.88); cost index 4.28 (cost 1.00)",
al-ain,Abu Dhabi,PROTECT,PROTECT,0.716,medium,"serves 18,652 (demand 0.93); no rent: cost neutral 0.5",
khalifa-city-a,Abu Dhabi,HOLD,PROTECT,0.667,low,"serves 16,679 (demand 0.83); no rent: cost neutral 0.5",demand weight ×0.75 → HOLD; cost weight ×1.25 → HOLD
mirdif-35,Dubai,HOLD,HOLD,0.637,low,"serves 5,490 (demand 0.27); cost index 2.67 (cost 1.00)",demand weight ×0.75 → PROTECT; cost weight ×1.25 → PROTECT
shahama,Abu Dhabi,HOLD,HOLD,0.613,low,"serves 9,865 (demand 0.49); cost index 1.23 (cost 0.73)",
shakhbout-city,Abu Dhabi,HOLD,HOLD,0.569,medium,"serves 12,743 (demand 0.64); no rent: cost neutral 0.5",
baniyas,Abu Dhabi,HOLD,HOLD,0.527,high,"serves 11,086 (demand 0.55); no rent: cost neutral 0.5",
al-falah,Abu Dhabi,HOLD,HOLD,0.488,high,"serves 9,510 (demand 0.48); no rent: cost neutral 0.5",
ministries-complex,Abu Dhabi,HOLD,HOLD,0.460,high,"serves 8,398 (demand 0.42); no rent: cost neutral 0.5",


In [4]:
ch = t[t.v3 != t.new]
print(len(ch), "of", len(t), "lounges change call:"); ch[["v3", "new", "composite", "reason"]]

8 of 23 lounges change call:


,v3,new,composite,reason
lounge,,,,
al-dhafra,HOLD,PROTECT,0.941,"serves 17,634 (demand 0.88); cost index 4.28 (..."
khalifa-city-a,HOLD,PROTECT,0.667,"serves 16,679 (demand 0.83); no rent: cost neu..."
ras-al-khaimah,PROTECT,HOLD,0.433,"serves 7,328 (demand 0.37); no rent: cost neut..."
al-taif-mall,PROTECT,HOLD,0.421,"serves 6,828 (demand 0.34); no rent: cost neut..."
mohammed-bin-zayed-city,SHRINK,HOLD,0.351,"serves 4,057 (demand 0.20); no rent: cost neut..."
al-barsha,HOLD,SHRINK,0.310,"serves 2,399 (demand 0.12); cost index 1.00 (c..."
jumeirah-park,HOLD,SHRINK,0.182,"serves 1,702 (demand 0.09); cost index 0.78 (c..."
city-walk,HOLD,SHRINK,0.066,"serves 2,175 (demand 0.11); cost index 0.52 (c..."


## 3. Cannibalisation what-if (display only)

Close each lounge, re-split: `absorbed` = the share of its served women its siblings would take over. Not scored.

In [5]:
cn = F[~F.not_scored][["emirate", "served_women", "absorbed_share"]].assign(absorbed_women=lambda x: x.served_women * x.absorbed_share)
cn["call"] = cn.index.map(lambda b: D[b].action)
cn.sort_values("absorbed_share", ascending=False).round(2)

,emirate,served_women,absorbed_share,absorbed_women,call
branch_id,,,,,
noya-plaza,Abu Dhabi,1385.72,0.30,419.15,SHRINK
westyas,Abu Dhabi,3160.09,0.27,850.06,SHRINK
khalifa-city-a,Abu Dhabi,16679.26,0.20,3308.14,PROTECT
mohammed-bin-zayed-city,Abu Dhabi,4056.92,0.19,784.20,HOLD
shahama,Abu Dhabi,9865.31,0.19,1879.86,HOLD
shakhbout-city,Abu Dhabi,12743.27,0.19,2381.51,HOLD
al-maqta,Abu Dhabi,6989.73,0.18,1236.37,HOLD
ministries-complex,Abu Dhabi,8397.89,0.16,1370.32,HOLD
al-falah,Abu Dhabi,9509.65,0.15,1447.75,HOLD


## 4. Composites against the lines

In [6]:
print(t.composite.describe().round(3))
print("within 0.05 of a line:", list(t.index[(abs(t.composite - scorecard.PROTECT_AT) < 0.05) | (abs(t.composite - scorecard.SHRINK_AT) < 0.05)]))
print("no cost index (cost scores neutral 0.5):", list(F.index[~F.not_scored & F.cost_index.isna()]))
print("confidence:", t.confidence.value_counts().to_dict())
dem = next(s for s in scorecard.SIGNALS if s.name == "demand")
print("with cost neutral (0.5), PROTECT needs served >=", round((2 * scorecard.PROTECT_AT - 0.5) * dem.best),
      "and SHRINK needs served <=", round((2 * scorecard.SHRINK_AT - 0.5) * dem.best))

count    23.000
mean      0.428
std       0.209
min       0.035
25%       0.318
50%       0.421
75%       0.548
max       0.941
Name: composite, dtype: float64
within 0.05 of a line: ['khalifa-city-a', 'mirdif-35', 'shahama', 'al-jada', 'nad-al-sheba', 'mohammed-bin-zayed-city', 'khaleej-al-arabi', 'al-barsha']
no cost index (cost scores neutral 0.5): ['al-ain', 'al-falah', 'al-jada', 'al-maqta', 'al-taif-mall', 'baniyas', 'delma', 'khaleej-al-arabi', 'khalifa-city-a', 'ministries-complex', 'mohammed-bin-zayed-city', 'ras-al-khaimah', 'shakhbout-city', 'zawaya-walk']
confidence: {'high': 8, 'low': 8, 'medium': 7}
with cost neutral (0.5), PROTECT needs served >= 16000 and SHRINK needs served <= 4000


## Open questions for review

1. **Most lounges are scored on demand alone.** 14 of 23 have no cost index, so cost scores a neutral 0.5. For them PROTECT needs 16,000+ women served and SHRINK 4,000 or fewer, while the 9 with a rent are judged on two signals. Is a mixed scorecard acceptable until rents arrive?
2. **8 of 23 lounges change call from v3.** al-dhafra and khalifa-city-a go HOLD → PROTECT. ras-al-khaimah and al-taif-mall go PROTECT → HOLD (they serve 7,328 and 6,828 and have no rent). mohammed-bin-zayed-city goes SHRINK → HOLD. al-barsha, jumeirah-park and city-walk go HOLD → SHRINK. New counts: PROTECT 3, HOLD 13, SHRINK 7.
3. **The three new Dubai SHRINKs** rest on Dubai's served demand (1.7k–2.4k women each) and cost indices of 1.00 or below. They stand or fall with the Dubai-competition question in `served_demand.ipynb`.
4. **al-dhafra's PROTECT (0.941)** leans on a cost index of 4.28 from a mall band (see `rent.ipynb`).
5. **Many calls sit near a line.** 8 lounges are within 0.05 of a line, and 8 are low confidence.
6. **Cannibalisation is display only.** It is highest at noya-plaza (0.30) and westyas (0.27), 0.01 at every Dubai lounge, and 0.00 at al-ain, al-dhafra, ras-al-khaimah and al-taif-mall. Is showing it, without scoring it, enough?